# 가중합과 선형방정식


## AI가 환자 데이터로 예측하는 핵심 원리

**가중합(Weighted Sum)은** AI가 여러 정보를 종합해서 하나의 예측값을 만드는 가장 기본적인 방법입니다.   
각 정보에 중요도(가중치)를 곱해서 모두 더하는 것입니다.



🎯 선형방정식 = 가중합 + 편향




### 편향(Bias)이란?

**편향(Bias)은** 모든 입력이 0일 때도 가지는 기본값입니다. 의료에서는 "기초 상태"라고 생각하면 됩니다.


In [ ]:
# 환자 일일 칼로리 소모량 계산 (여러 특성 + 각기 다른 가중치)
activity_data = np.array([45, 8, 30, 2, 6])  # [나이, 수면시간, 운동시간, 계단오르기, 걷기시간]
calorie_weights = np.array([0.8, 1.5, 0.6, 2.0, 0.3])  # 각 활동별 칼로리 계수
basal_metabolic_rate = 1400  # 기초대사율 (편향)

print("=== 환자 일일 칼로리 소모량 계산 ===")

# 가중합 계산
total_activity_calories = np.dot(activity_data, calorie_weights) # 스칼라
print(f"가중합: {total_activity_calories}")

# 선형방정식: 가중합 + 편향
total_daily_calories = total_activity_calories + basal_metabolic_rate

print(f"최종 일일 소모량: {total_activity_calories} + {basal_metabolic_rate} = {total_daily_calories}kcal")

if total_daily_calories > 2500:
    print("💚 우수")
elif total_daily_calories > 2000:
    print("💛 보통")
else:
    print("🧡 부족")



=== 환자 일일 칼로리 소모량 계산 ===
가중합: 71.8
최종 일일 소모량: 71.8 + 1400 = 1471.8kcal
🧡 부족


💡 편향(기초대사율)은 '잠만 자도 소모되는 기본 칼로리'를 나타냅니다.  
💡 활동을 전혀 하지 않아도 1400kcal는 소모됩니다

## 여러 환자의 약물 투여량 계산


In [ ]:
# 여러 환자의 약물 투여량 계산
patients_info = np.array([
    [70, 45],   # 환자1: 체중 70kg, 나이 45세
    [55, 32],   # 환자2: 체중 55kg, 나이 32세
    [85, 60]    # 환자3: 체중 85kg, 나이 60세
])

dosage_weights = np.array([0.5, 0.1])  # [체중 계수, 나이 계수]
base_dosage = 10  # 기본 용량 (편향)

print("=== 개별 맞춤 약물 투여량 계산 ===")

# 모든 환자의 가중합을 한 번에 계산 (가중합 + 편향)
total_dosages = np.dot(patients_info, dosage_weights) + base_dosage # 벡터

# 각 환자별 결과 출력
for i, (patient, dosage) in enumerate(zip(patients_info, total_dosages)):
    weight, age = patient

    print(f"환자{i+1}: 체중 {weight}kg, 나이 {age}세 → 투여량: {dosage}mg")

=== 개별 맞춤 약물 투여량 계산 ===
환자1: 체중 70kg, 나이 45세 → 투여량: 49.5mg
환자2: 체중 55kg, 나이 32세 → 투여량: 40.7mg
환자3: 체중 85kg, 나이 60세 → 투여량: 58.5mg


## AI가 가중치를 학습하는 과정

### 최적 가중치 찾기


In [ ]:
actual_patient = np.array([65, 50])  # [체중, 나이]
base_dosage = 10                    # 기본 용량 (mg)
correct_dosage = 45                  # 정답 투여량

# AI가 시도해보는 여러 가중치
trial_weights = [
    np.array([0.3, 0.2]),  # 시도 1
    np.array([0.5, 0.1]),  # 시도 2
    np.array([0.6, 0.05])  # 시도 3
]

print(f"환자 정보: 체중 {actual_patient[0]}kg, 나이 {actual_patient[1]}세")
print(f"정답 투여량: {correct_dosage}mg")


환자 정보: 체중 65kg, 나이 50세
정답 투여량: 45mg


In [ ]:
# 최적 가중치 찾기 위한 변수 초기화
best_weights = None
smallest_error = 1000 # 기본 오차

# 각 가중치 조합을 테스트
for i, weights in enumerate(trial_weights):
    # 가중합으로 예측값 계산
    weighted_sum = np.dot(actual_patient, weights)
    predicted_dosage = weighted_sum + base_dosage  # 최종 예측값

    # 오차
    error = abs(correct_dosage - predicted_dosage)  # 절댓값 함수 (음수를 양수로 변환)

    print(f"\n시도 {i+1}: 가중치 {weights}")
    print(f"  예측 투여량: {predicted_dosage:.1f}mg")
    print(f"  오차: {error:.1f}mg")

    # 가장 작은 오차를 가진 가중치 저장
    if error < smallest_error: # 기존 에러값보다 현재 에러값이 작으면 현재 에러값으로 치환
        smallest_error = error
        best_weights = weights

print(f"\n최적 가중치: {best_weights} (오차: {smallest_error:.1f}mg)")


시도 1: 가중치 [0.3 0.2]
  예측 투여량: 39.5mg
  오차: 5.5mg

시도 2: 가중치 [0.5 0.1]
  예측 투여량: 47.5mg
  오차: 2.5mg

시도 3: 가중치 [0.6  0.05]
  예측 투여량: 51.5mg
  오차: 6.5mg

최적 가중치: [0.5 0.1] (오차: 2.5mg)


## 🤔 실습 문제

환자의 수술 위험도를 계산해보세요.
- 위험도 = (나이 × 0.8) + (기존질환수 × 15) + (BMI × 2.0) + 기본위험도
- 기본위험도: 20점

**위험도 등급 및 권고사항**
- 40점 미만: ```낮음``` → "수술 진행 가능" 출력
- 40-59점: ```보통``` → "수술 진행 가능" 출력
- 60-79점: ```높음``` → "추가 검사 필요" 출력
- 80점 이상: ```매우 높음``` → "추가 검사 및 위험도 감소 조치 필요" 출력

In [ ]:
# 환자 정보
patient_info = np.array([58, 2, 28.5])  # [나이, 기존질환수, BMI]


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# @title 정답

# 환자 정보
patient_info = np.array([58, 2, 28.5])  # [나이, 기존질환수, BMI]

# 가중치
risk_weights = np.array([0.8, 15, 2.0])  # [나이 계수, 질환 계수, BMI 계수]
base_risk = 20  # 기본 위험도 (편향)

print("=== 수술 위험도 평가 ===")
print(f"환자 정보: 나이 {patient_info[0]}세, 기존질환 {patient_info[1]}개, BMI {patient_info[2]}")

# 가중합 계산 (NumPy 내적 사용)
weighted_sum = np.dot(patient_info, risk_weights)

# 선형방정식: 가중합 + 편향
total_risk = weighted_sum + base_risk

print(f"위험도 점수: {total_risk}점")

# 위험도 등급 분류
if total_risk < 40:
    risk_level = "낮음"
    recommendation = "수술 진행 가능"
elif total_risk < 60:
    risk_level = "보통"
    recommendation = "수술 진행 가능"
elif total_risk < 80:
    risk_level = "높음"
    recommendation = "추가 검사 필요"
else:
    risk_level = "매우 높음"
    recommendation = "추가 검사 및 위험도 감소 조치 필요"

print(f"수술 위험도: {risk_level}")
print(f"권고사항: {recommendation}")

=== 수술 위험도 평가 ===
환자 정보: 나이 58.0세, 기존질환 2.0개, BMI 28.5
위험도 점수: 153.4점
수술 위험도: 매우 높음
권고사항: 추가 검사 및 위험도 감소 조치 필요
